<a href="https://colab.research.google.com/github/Touseeq99/ML_MODELS_SCRATCH/blob/main/FineTuning_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install unsloth
!pip install datasets transformers accelerate evaluate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 6.7 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 373.9/373.9 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.3/289.3 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.9/122.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 768.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.5/170.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2

In [ ]:
from unsloth import FastLanguageModel
import torch

model1, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-1.5B-Instruct",
    max_seq_length = 2048,
    dtype = torch.float16,
    load_in_4bit = False,
)

==((====))==  Unsloth 2025.12.5: Fast Qwen2 patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [ ]:
from datasets import load_dataset

dataset = load_dataset("gsm8k", "main", split="test[:50]")


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

In [ ]:
def generate_answer(question):
    prompt = f"""
Solve the math problem step by step and give the final answer.

Question: {question}
Answer:
"""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.3
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


In [ ]:
correct = 0

for sample in dataset:
    pred = generate_answer(sample["question"])
    if sample["answer"].split("####")[-1].strip() in pred:
        correct += 1

print("Baseline Accuracy:", correct / len(dataset))


Baseline Accuracy: 0.58


In [ ]:
def format_prompt(example):
    return {
        "text": f"""
Solve the math problem step by step and give the final answer.

Question: {example['question']}
Answer: {example['answer']}
"""
    }

train_data = load_dataset("gsm8k", "main", split="train[:500]")
train_data = train_data.map(format_prompt)


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
model = FastLanguageModel.get_peft_model(
    model1,
    r = 16,
    lora_alpha = 32,
    lora_dropout = 0.05,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"],
)


Unsloth: Already have LoRA adapters! We shall skip this step.


In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# Assuming max_seq_length = 2048 from the model loading in a previous cell
max_seq_length = 2048

# Tokenize the dataset

def tokenize_function(examples):
    # The tokenizer is globally available from a previous cell
    tokenized_inputs = tokenizer(
        examples["text"],
        padding=True, # DataCollator will handle padding
        truncation=True,
        max_length=max_seq_length,
    )
    # For causal language modeling, labels are usually the input_ids
    tokenized_inputs["labels"] = tokenized_inputs["input_ids"].copy()
    return tokenized_inputs

# Apply tokenization to the training data
train_data_tokenized = train_data.map(
    tokenize_function,
    batched=True,
    remove_columns=["question", "answer", "text"], # Remove original columns no longer needed
    desc="Tokenizing training data",
)

# Initialize a data collator to handle padding for batches
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model = model1,
    train_dataset = train_data_tokenized, # Use the tokenized dataset
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = True,
        logging_steps = 10,
        output_dir = "./outputs",
        remove_unused_columns=False, # Prevent Trainer from auto-removing columns
    ),
    data_collator=data_collator, # Add data collator
)

trainer.train()

Tokenizing training data:   0%|          | 0/500 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 2 | Total steps = 126
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 4,358,144 of 1,548,072,448 (0.28% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,0.583000
20,0.478800
30,0.397900
40,0.354400
50,0.335300
60,0.310500
70,0.311900
80,0.288100
90,0.304600
100,0.265300


TrainOutput(global_step=126, training_loss=0.34536786117250956, metrics={'train_runtime': 329.1657, 'train_samples_per_second': 3.038, 'train_steps_per_second': 0.383, 'total_flos': 4377946844160000.0, 'train_loss': 0.34536786117250956, 'epoch': 2.0})